# C05 — Image Segmentation: U-Net from Scratch

> **Audience**: PhD students · **Framework**: PyTorch · **Data**: synthetic + Hugging Face

Segmentation assigns a class label to **every pixel** rather than the whole image
or a bounding box. This requires architectures that preserve spatial resolution
throughout the network.

**What this notebook builds**
1. U-Net encoder (contracting path) with skip connections — from scratch
2. U-Net decoder (expanding path) — from scratch
3. Full U-Net model
4. Pixel-wise loss and training on synthetic data
5. SegFormer inference via Hugging Face — modern transformer-based segmentation

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# 1) U-Net Architecture

**Origin and intuition** (Ronneberger et al., 2015)
U-Net was designed for biomedical image segmentation where:
- Training data is scarce (you cannot use ImageNet-scale datasets)
- Precise spatial localisation is required (pixel-level accuracy)

The "U" shape comes from its symmetric encoder–decoder structure:

```
Encoder (contracting path)
  Conv block → MaxPool   32 ch, H×W
  Conv block → MaxPool   64 ch, H/2×W/2
  Conv block → MaxPool   128 ch, H/4×W/4
  Conv block (bottleneck) 256 ch, H/8×W/8

Decoder (expanding path)
  Upsample + skip concat + Conv block   128 ch, H/4×W/4
  Upsample + skip concat + Conv block   64 ch, H/2×W/2
  Upsample + skip concat + Conv block   32 ch, H×W
  1×1 Conv → num_classes
```

**Why skip connections?**
The encoder's downsampling discards fine spatial detail that is needed for
accurate per-pixel prediction. Skip connections copy the encoder feature maps
directly to the corresponding decoder stage, giving the decoder access to
high-resolution features that were never downsampled.

In [ ]:
def conv_block(in_channels: int, out_channels: int) -> nn.Sequential:
    """
    Two 3×3 conv layers with BatchNorm and ReLU — the basic U-Net building block.

    Design decision: BatchNorm after each conv stabilises training and allows
    higher LRs. The original U-Net did not use BN; modern variants do.

    Args:
        in_channels  : Input channel count
        out_channels : Output channel count

    Returns:
        Sequential: Conv → BN → ReLU → Conv → BN → ReLU
    """
    return nn.Sequential(
        # (batch_num, in_channels, h, w) → (batch_num, out_channels, h, w)
        nn.Conv2d(in_channels,  out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        # (batch_num, out_channels, h, w) → (batch_num, out_channels, h, w)
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
    )


# ── Verify conv_block shape ────────────────────────────────────────────────────
block_test = conv_block(3, 32)
x_test = torch.zeros(2, 3, 64, 64)
with torch.no_grad():
    out_test = block_test(x_test)
print(f"conv_block  in : {x_test.shape}")   # (2, 3, 64, 64)
print(f"conv_block out : {out_test.shape}") # (2, 32, 64, 64)  ← spatial unchanged

# 2) Encoder (Contracting Path)

The encoder progressively:
1. Extracts features with `conv_block`
2. Stores the feature map as a skip connection
3. Downsamples with MaxPool to build a larger receptive field

After each MaxPool the channel count doubles, following the U-Net convention
of trading spatial resolution for feature richness.

In [ ]:
class UNetEncoder(nn.Module):
    """
    U-Net contracting path.

    At each stage: apply conv_block, save skip connection, then MaxPool.
    Returns the bottleneck features plus all skip connections.
    """

    def __init__(self, in_channels: int = 3, base_channels: int = 32) -> None:
        super().__init__()

        c = base_channels

        # Four encoder stages — each doubles channels
        # Stage 1: (batch_num, in_channels, H, W)   → (batch_num, c, H, W)
        self.enc1 = conv_block(in_channels, c)
        # Stage 2: (batch_num, c, H/2, W/2)         → (batch_num, 2c, H/2, W/2)
        self.enc2 = conv_block(c,  c * 2)
        # Stage 3: (batch_num, 2c, H/4, W/4)        → (batch_num, 4c, H/4, W/4)
        self.enc3 = conv_block(c * 2, c * 4)
        # Bottleneck: (batch_num, 4c, H/8, W/8)     → (batch_num, 8c, H/8, W/8)
        self.enc4 = conv_block(c * 4, c * 8)

        # MaxPool between stages — halves spatial dimensions
        self.pool = nn.MaxPool2d(kernel_size=2)

    def forward(self, x: torch.Tensor) -> tuple:
        """
        Returns:
            bottleneck : (batch_num, 8c, H/8, W/8)
            skips      : list of feature maps [(batch_num, c, H, W),
                                               (batch_num, 2c, H/2, W/2),
                                               (batch_num, 4c, H/4, W/4)]
        """
        # (batch_num, in_channels, H, W) → (batch_num, c, H, W)
        s1 = self.enc1(x)
        # (batch_num, c, H, W) → (batch_num, 2c, H/2, W/2)
        s2 = self.enc2(self.pool(s1))
        # (batch_num, 2c, H/2, W/2) → (batch_num, 4c, H/4, W/4)
        s3 = self.enc3(self.pool(s2))
        # (batch_num, 4c, H/4, W/4) → (batch_num, 8c, H/8, W/8)
        bottleneck = self.enc4(self.pool(s3))

        return bottleneck, [s1, s2, s3]


# ── Shape test ─────────────────────────────────────────────────────────────────
enc = UNetEncoder(in_channels=3, base_channels=32)
x_enc = torch.zeros(2, 3, 128, 128)
with torch.no_grad():
    bn, skips = enc(x_enc)
print(f"Input         : {x_enc.shape}")          # (2, 3, 128, 128)
print(f"Bottleneck    : {bn.shape}")              # (2, 256, 16, 16)
for i, s in enumerate(skips):
    print(f"Skip {i+1}        : {s.shape}")

# 3) Decoder (Expanding Path)

The decoder mirrors the encoder: at each stage it
1. **Upsamples** the feature map (using bilinear interpolation — smoother than transposed conv)
2. **Concatenates** the corresponding skip connection from the encoder
3. Applies a `conv_block` to fuse the upsampled and skip features

**Why concatenate and not add?**
Concatenation preserves both the low-level spatial detail (from the skip)
and the high-level semantic content (from the upsampled bottleneck).
Addition would require them to be in the same feature space, which they
are not at this point in training.

In [ ]:
class UNetDecoder(nn.Module):
    """
    U-Net expanding path.

    At each stage: upsample → concatenate skip connection → conv_block.
    The skip connection doubles the channel count before conv_block reduces it.
    """

    def __init__(self, base_channels: int = 32) -> None:
        super().__init__()

        c = base_channels

        # Three decoder stages — each halves channels
        # Up1: (batch_num, 8c+4c, H/4, W/4) → (batch_num, 4c, H/4, W/4)
        self.dec1 = conv_block(c * 8 + c * 4, c * 4)
        # Up2: (batch_num, 4c+2c, H/2, W/2) → (batch_num, 2c, H/2, W/2)
        self.dec2 = conv_block(c * 4 + c * 2, c * 2)
        # Up3: (batch_num, 2c+c, H, W)       → (batch_num, c, H, W)
        self.dec3 = conv_block(c * 2 + c,     c)

    def forward(
        self, bottleneck: torch.Tensor, skips: list
    ) -> torch.Tensor:
        """
        Args:
            bottleneck : (batch_num, 8c, H/8, W/8)
            skips      : [s1, s2, s3] from the encoder

        Returns:
            out : (batch_num, c, H, W) — full resolution feature map
        """
        s1, s2, s3 = skips

        # Stage 1: upsample bottleneck → match s3 spatial size, then concat
        # (batch_num, 8c, H/8, W/8) → (batch_num, 8c, H/4, W/4)
        up1 = F.interpolate(bottleneck, size=s3.shape[2:], mode="bilinear", align_corners=False)
        # (batch_num, 8c+4c, H/4, W/4) → (batch_num, 4c, H/4, W/4)
        d1  = self.dec1(torch.cat([up1, s3], dim=1))

        # Stage 2
        # (batch_num, 4c, H/4, W/4) → (batch_num, 4c, H/2, W/2)
        up2 = F.interpolate(d1, size=s2.shape[2:], mode="bilinear", align_corners=False)
        # (batch_num, 4c+2c, H/2, W/2) → (batch_num, 2c, H/2, W/2)
        d2  = self.dec2(torch.cat([up2, s2], dim=1))

        # Stage 3
        # (batch_num, 2c, H/2, W/2) → (batch_num, 2c, H, W)
        up3 = F.interpolate(d2, size=s1.shape[2:], mode="bilinear", align_corners=False)
        # (batch_num, 2c+c, H, W) → (batch_num, c, H, W)
        d3  = self.dec3(torch.cat([up3, s1], dim=1))

        return d3


# ── Shape test ─────────────────────────────────────────────────────────────────
dec = UNetDecoder(base_channels=32)
with torch.no_grad():
    out_dec = dec(bn, skips)
print(f"Decoder output : {out_dec.shape}")  # (2, 32, 128, 128)  ← full resolution

# 4) Full U-Net Model

Combine encoder, decoder, and add a 1×1 convolution to map the final
feature channels to class logits.

**1×1 convolution as the final layer**
A 1×1 conv applies an independent linear transformation at each pixel.
It maps `base_channels` features to `num_classes` logits — one logit per class
per pixel — without any spatial mixing.

In [ ]:
class UNet(nn.Module):
    """
    Full U-Net for semantic segmentation.

    Input  : (batch_num, in_channels, H, W)
    Output : (batch_num, num_classes, H, W)  — raw logits per pixel
    """

    def __init__(
        self, in_channels: int = 3, num_classes: int = 2, base_channels: int = 32
    ) -> None:
        super().__init__()

        self.encoder = UNetEncoder(in_channels, base_channels)
        self.decoder = UNetDecoder(base_channels)

        # Final 1×1 conv: map features to per-pixel class logits
        # (batch_num, base_channels, H, W) → (batch_num, num_classes, H, W)
        self.seg_head = nn.Conv2d(base_channels, num_classes, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, in_channels, H, W) → bottleneck + skips
        bottleneck, skips = self.encoder(x)
        # bottleneck + skips → (batch_num, base_channels, H, W)
        features = self.decoder(bottleneck, skips)
        # (batch_num, base_channels, H, W) → (batch_num, num_classes, H, W)
        return self.seg_head(features)


# ── Full shape dry-run ─────────────────────────────────────────────────────────
unet = UNet(in_channels=3, num_classes=4, base_channels=32)
x_u  = torch.zeros(2, 3, 128, 128)
with torch.no_grad():
    logits_u = unet(x_u)

print(f"U-Net input  : {x_u.shape}")       # (2, 3, 128, 128)
print(f"U-Net output : {logits_u.shape}")  # (2, 4, 128, 128)  ← same H×W, 4 classes
print(f"Parameters   : {sum(p.numel() for p in unet.parameters() if p.requires_grad):,}")

# 5) Synthetic Dataset and Training

**Synthetic segmentation task**
We generate images with 3 coloured geometric shapes on a black background.
Each pixel belongs to one of 4 classes: background, circle, rectangle, triangle.

This keeps the notebook self-contained (no large downloads) while exercising
all the real training machinery: pixel-wise loss, class imbalance, spatial accuracy.

**Loss function: SparseCategoricalCrossEntropy**
Each pixel's target is a single integer class index (not one-hot).
`F.cross_entropy` with a (N, C, H, W) input and (N, H, W) long target
handles this directly via `SparseCrossEntropyLoss`.

**Class imbalance**
Background pixels dominate in most segmentation datasets.
We address this with class weights inversely proportional to frequency.

In [ ]:
class ShapeSegDataset(Dataset):
    """
    Synthetic dataset: 3-channel RGB images with geometric shapes.

    Classes:
        0 = background (black)
        1 = circle (red)
        2 = rectangle (green)
        3 = triangle (blue)
    """

    def __init__(self, num_samples: int = 200, img_size: int = 128) -> None:
        self.num_samples = num_samples
        self.img_size    = img_size
        np.random.seed(0)
        self.data = [self._generate() for _ in range(num_samples)]

    def _generate(self) -> tuple:
        H = W = self.img_size
        img  = np.zeros((H, W, 3), dtype=np.float32)
        mask = np.zeros((H, W),    dtype=np.int64)

        for shape_id, colour in enumerate([[1,0,0], [0,1,0], [0,0,1]], start=1):
            cx = np.random.randint(20, W-20)
            cy = np.random.randint(20, H-20)
            r  = np.random.randint(8, 20)

            if shape_id == 1:  # circle
                Y, X = np.ogrid[:H, :W]
                m = (X-cx)**2 + (Y-cy)**2 <= r**2
            elif shape_id == 2:  # rectangle
                x1, x2 = cx-r, cx+r
                y1, y2 = cy-r, cy+r
                m = np.zeros((H, W), bool)
                m[max(0,y1):y2, max(0,x1):x2] = True
            else:  # triangle (row approximation)
                m = np.zeros((H, W), bool)
                for dy in range(r):
                    x_start = max(0, cx-(r-dy))
                    x_end   = min(W, cx+(r-dy)+1)
                    row     = cy+dy
                    if 0 <= row < H:
                        m[row, x_start:x_end] = True

            img[m]  = colour
            mask[m] = shape_id

        # (H, W, 3) → (3, H, W)
        return torch.tensor(img).permute(2, 0, 1), torch.tensor(mask)

    def __len__(self):  return self.num_samples
    def __getitem__(self, i): return self.data[i]


# ── Build loaders ──────────────────────────────────────────────────────────────
dataset      = ShapeSegDataset(num_samples=300, img_size=128)
train_ds, val_ds = torch.utils.data.random_split(dataset, [240, 60])
train_dl = DataLoader(train_ds, batch_size=8, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=8, shuffle=False)

img_sample, mask_sample = dataset[0]
fig, axes = plt.subplots(1, 2, figsize=(8, 3))
axes[0].imshow(img_sample.permute(1, 2, 0).numpy()); axes[0].set_title("Image"); axes[0].axis("off")
axes[1].imshow(mask_sample.numpy(), cmap="tab10", vmin=0, vmax=3); axes[1].set_title("Mask (0=bg,1=circle,2=rect,3=tri)"); axes[1].axis("off")
plt.tight_layout(); plt.show()


def train_unet(model, train_dl, val_dl, n_epochs=10, lr=1e-3):
    """Trains U-Net with pixel-wise cross-entropy loss."""
    model.to(DEVICE)
    # Weight background class lower — it dominates by pixel count
    class_weights = torch.tensor([0.2, 1.0, 1.0, 1.0]).to(DEVICE)
    criterion  = nn.CrossEntropyLoss(weight=class_weights)
    optimizer  = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    history    = {"train_loss": [], "val_miou": []}

    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss = 0.0
        for imgs, masks in train_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            optimizer.zero_grad()
            # (batch_num, 3, H, W) → (batch_num, num_classes, H, W)
            logits = model(imgs)
            # cross_entropy expects (N, C, H, W) logits and (N, H, W) targets
            loss = criterion(logits, masks)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * imgs.size(0)
        scheduler.step()

        # mIoU on validation
        model.eval()
        iou_per_class = torch.zeros(4)
        with torch.no_grad():
            for imgs, masks in val_dl:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                # (batch_num, num_classes, H, W) → (batch_num, H, W)
                preds = model(imgs).argmax(dim=1)
                for c in range(4):
                    inter = ((preds == c) & (masks == c)).sum().float()
                    union = ((preds == c) | (masks == c)).sum().float()
                    iou_per_class[c] += (inter / (union + 1e-8)).cpu()
        miou = (iou_per_class / len(val_dl)).mean().item()

        tl = running_loss / len(train_dl.dataset)
        history["train_loss"].append(tl)
        history["val_miou"].append(miou)
        print(f"Epoch {epoch:02d}/{n_epochs} | loss={tl:.4f} | val_mIoU={miou:.3f}")

    return history


torch.manual_seed(0)
unet_model = UNet(in_channels=3, num_classes=4, base_channels=32)
history_seg = train_unet(unet_model, train_dl, val_dl, n_epochs=10)

In [ ]:
# ── Visualise predictions ──────────────────────────────────────────────────────
unet_model.eval()
imgs_v, masks_v = next(iter(val_dl))
imgs_v = imgs_v.to(DEVICE)

with torch.no_grad():
    # (batch_num, num_classes, H, W) → (batch_num, H, W)
    preds_v = unet_model(imgs_v).argmax(dim=1).cpu()

fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for col in range(4):
    axes[0, col].imshow(imgs_v[col].cpu().permute(1,2,0)); axes[0, col].axis("off")
    axes[1, col].imshow(masks_v[col], cmap="tab10", vmin=0, vmax=3); axes[1, col].axis("off")
    axes[2, col].imshow(preds_v[col], cmap="tab10", vmin=0, vmax=3); axes[2, col].axis("off")

axes[0,0].set_ylabel("Input", fontsize=9)
axes[1,0].set_ylabel("Ground truth", fontsize=9)
axes[2,0].set_ylabel("Prediction", fontsize=9)
plt.suptitle("U-Net segmentation results")
plt.tight_layout(); plt.show()

# Loss and mIoU curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3))
ax1.plot(history_seg["train_loss"]); ax1.set_title("Training loss"); ax1.set_xlabel("Epoch")
ax2.plot(history_seg["val_miou"]);   ax2.set_title("Val mIoU");      ax2.set_xlabel("Epoch")
plt.tight_layout(); plt.show()

# 6) Modern Segmentation: SegFormer

SegFormer (Xie et al., 2021) replaces the U-Net's CNN backbone with a
hierarchical Vision Transformer, and uses a lightweight MLP decoder instead
of the symmetric skip-connection decoder.

**Key improvements over U-Net**
- Mix-Transformer (MiT) encoder produces multi-scale features natively
- No positional encoding → generalises to different resolutions at test time
- All-MLP decoder is surprisingly effective (0.4M params vs U-Net's multi-scale fusion)

**Models available on Hugging Face**
```
nvidia/segformer-b0-finetuned-ade-512-512   # lightest (3.7M params)
nvidia/segformer-b5-finetuned-ade-640-640   # heaviest, best mIoU
```

In [ ]:
# Install: !pip install transformers --quiet

from transformers import SegformerImageProcessor, SegformerForSemanticSegmentation
import requests
from PIL import Image
from io import BytesIO

# Load pretrained SegFormer-B0 (lightest variant)
processor = SegformerImageProcessor.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512"
)
segformer = SegformerForSemanticSegmentation.from_pretrained(
    "nvidia/segformer-b0-finetuned-ade-512-512"
).to(DEVICE)

print(f"SegFormer parameters: {sum(p.numel() for p in segformer.parameters()):,}")
print(f"Number of ADE20K classes: {segformer.config.num_labels}")

# ── Inference on a sample image ────────────────────────────────────────────────
url    = "http://images.cocodataset.org/val2017/000000039769.jpg"
resp   = requests.get(url, timeout=15)
image  = Image.open(BytesIO(resp.content)).convert("RGB")

# Preprocess: resize, normalise, convert to tensor
inputs = processor(images=image, return_tensors="pt").to(DEVICE)
# inputs["pixel_values"]: (1, 3, 512, 512)

with torch.no_grad():
    outputs = segformer(**inputs)
    # outputs.logits: (1, num_classes, H/4, W/4)  — SegFormer predicts at 1/4 scale
    logits = outputs.logits

# Upsample to original image size for display
# (1, num_classes, H/4, W/4) → (1, num_classes, H, W)
logits_up = F.interpolate(
    logits, size=(image.height, image.width), mode="bilinear", align_corners=False
)
# (1, H, W)
pred_mask = logits_up.argmax(dim=1).squeeze(0).cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image)
axes[0].set_title("Input image"); axes[0].axis("off")
axes[1].imshow(pred_mask, cmap="tab20")
axes[1].set_title("SegFormer-B0 prediction (ADE20K classes)"); axes[1].axis("off")
plt.tight_layout(); plt.show()

print(f"Logits shape (at 1/4 scale)      : {logits.shape}")
print(f"Upsampled to original resolution : {logits_up.shape}")

# Summary

| Architecture | Skip connections | Decoder | Best for |
|---|---|---|---|
| U-Net (ours) | Encoder–decoder concat | Symmetric upsampling | Small datasets, medical imaging |
| SegFormer | None (MiT handles scales) | Lightweight MLP | Large-scale benchmarks |
| Mask2Former | Cross-attention to queries | Transformer decoder | Panoptic / instance seg |

**Key mIoU metric**
$$\text{mIoU} = \frac{1}{C} \sum_{c=1}^{C} \frac{TP_c}{TP_c + FP_c + FN_c}$$

**Research observations**
- U-Net's symmetric skip-connection design is the template for modern architectures —
  diffusion model denoisers (c09) use the same encoder–decoder pattern.
- Class imbalance is severe in segmentation (background >> objects);
  weighted cross-entropy or Dice loss are standard mitigations.
- SegFormer's 1/4-scale prediction + bilinear upsample is sufficient for
  evaluation; however, methods like SETR use full-resolution prediction for
  sharper edges at the cost of compute.